# Final supervised ML pipeline audit — selection bias and freeze gate

Zakres audytu jest ograniczony do train 2011–2020. External development
validation 2021–2022 jest użyte wyłącznie jako **minimalny indeks** do
potwierdzenia zaakceptowanego `N=3 547`: nie wczytujemy jego target labels,
financial features ani rozkładów. Lata 2023–2024 nie są wczytywane.

Audyt nie trenuje modeli predykcyjnych. Jedyny estymowany model to jawnie
oznaczony model mechanizmu dostępności targetu, potrzebny wyłącznie do oceny
wykonalności IPW. Nie używa on wartości targetu ani wyników przyszłych modeli.

Frozen target, universe i raw `X_t` są wejściami tylko do odczytu. Ten notebook
wydaje werdykt freeze-gate, ale nie tworzy manifestu freeze i niczego
automatycznie nie zamraża.


In [1]:
from pathlib import Path
import csv
import hashlib
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 72)

def project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs/x_t_pit_v1_freeze_manifest.yaml").is_file():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu głównego projektu.")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.modeling.preprocessing import (
    FEATURE_BLOCKS,
    FROZEN_BLOCK_COMPARISONS,
    PreprocessingPolicy,
    features_for_blocks,
)
from src.modeling.temporal_cv import MAIN_EXPANDING_WINDOW_FOLDS

FEATURES = list(features_for_blocks(("L", "D", "R")))
VALUE_COLUMNS = [f"{feature}_value" for feature in FEATURES]
ACCEPTED_X_STATUSES = {"available_core", "partially_available"}
RANDOM_SEED = 20260818

PATHS = {
    "universe": ROOT / "data/processed/research_universe_pit.csv",
    "target": ROOT / "data/interim/target_candidate_v2_pit_b.csv",
    "x_t": ROOT / "data/processed/x_t_pit_v1_raw.csv",
    "target_application": ROOT / "data/processed/research_universe_pit_v1_1_0_target_pit_b_v1_0_0.csv",
}
EXPECTED = {
    "universe": ("a449c8145d1f46f954f12b1dfc079bb0b367c4f7f5edf3332a983ad7c1fb8182", 103099, 81),
    "target": ("473aa403dfd15822a15ce985f7698efe4a4e3a66bcf30b7634f0ca646805e0ff", 26917, 802),
    "x_t": ("0f1b35b9ffbb1fb1c1cdfb7dff12e3efd8fb38f60b33407ff2b2a8fb6b88397f", 64901, 1072),
    "target_application": ("ea42eb43018b2c8e238e2c4757260bb692e27edd5429628e28892f360f0f7f7d", 64901, 832),
}

def fingerprint_csv(path: Path) -> dict:
    digest = hashlib.sha256()
    byte_count = 0
    newline_count = 0
    with path.open("rb") as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
            byte_count += len(chunk)
            newline_count += chunk.count(b"\n")
    with path.open(newline="", encoding="utf-8") as handle:
        columns = len(next(csv.reader(handle)))
    return {
        "sha256": digest.hexdigest(),
        "rows": max(newline_count - 1, 0),
        "columns": columns,
        "MiB": round(byte_count / 1024**2, 1),
    }

def read_years(path: Path, usecols: list[str], first_year: int, last_year: int) -> pd.DataFrame:
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=10_000, low_memory=False):
        years = pd.to_numeric(chunk["feature_year"], errors="coerce")
        parts.append(chunk.loc[years.between(first_year, last_year)].copy())
    return pd.concat(parts, ignore_index=True)

def show(title: str, value: pd.DataFrame | pd.Series) -> None:
    print(f"\n{title}")
    print("=" * len(title))
    print(value.to_string())

fingerprints = {name: fingerprint_csv(path) for name, path in PATHS.items()}
for name, actual in fingerprints.items():
    expected_hash, expected_rows, expected_columns = EXPECTED[name]
    assert actual["sha256"] == expected_hash, f"Hash mismatch: {name}"
    assert actual["rows"] == expected_rows, f"Row mismatch: {name}"
    assert actual["columns"] == expected_columns, f"Column mismatch: {name}"

x_columns = [
    "research_universe_company_year_id", "feature_year", "split",
    "membership_status", "x_t_status", "x_t_status_reason", "research_sector",
    "xbrl_submission_available", "statement_scope_xbrl_available",
    "statement_scope_xbrl_status", "registrant_role_resolved",
    "joint_filing_flag", "economic_group_id",
] + VALUE_COLUMNS
target_columns = [
    "research_universe_company_year_id", "feature_year", "target_status",
    "target_candidate_v2_pit_b",
]
x_train = read_years(PATHS["x_t"], x_columns, 2011, 2020)
target_train = read_years(PATHS["target_application"], target_columns, 2011, 2020).rename(
    columns={"feature_year": "target_feature_year"}
)
eligible_train = x_train.merge(
    target_train,
    on="research_universe_company_year_id",
    how="left",
    validate="one_to_one",
)
eligible_train = eligible_train.loc[
    eligible_train["membership_status"].eq("eligible")
    & eligible_train["split"].eq("train")
].copy()
assert eligible_train["feature_year"].eq(eligible_train["target_feature_year"]).all()
for value_column in VALUE_COLUMNS:
    eligible_train[value_column] = pd.to_numeric(eligible_train[value_column], errors="raise")
eligible_train["available_feature_count"] = eligible_train[VALUE_COLUMNS].notna().sum(axis=1)
eligible_train["target_available"] = eligible_train["target_status"].eq("available").astype(int)
eligible_train["x_usable"] = eligible_train["x_t_status"].isin(ACCEPTED_X_STATUSES).astype(int)
eligible_train["main_selected"] = (
    eligible_train["target_available"].eq(1) & eligible_train["x_usable"].eq(1)
)

minimal_x_columns = [
    "research_universe_company_year_id", "feature_year", "split",
    "membership_status", "x_t_status",
]
minimal_target_columns = [
    "research_universe_company_year_id", "feature_year", "target_status",
]
x_external_index = read_years(PATHS["x_t"], minimal_x_columns, 2021, 2022)
target_external_index = read_years(
    PATHS["target_application"], minimal_target_columns, 2021, 2022
).rename(columns={"feature_year": "target_feature_year"})
external_index = x_external_index.merge(
    target_external_index,
    on="research_universe_company_year_id",
    how="left",
    validate="one_to_one",
)
external_index = external_index.loc[
    external_index["membership_status"].eq("eligible")
    & external_index["split"].eq("validation")
    & external_index["target_status"].eq("available")
    & external_index["x_t_status"].isin(ACCEPTED_X_STATUSES),
    ["research_universe_company_year_id", "feature_year"],
].copy()

main_train = eligible_train.loc[eligible_train["main_selected"]].copy()
assert len(eligible_train) == 47_938
assert len(main_train) == 19_671
assert len(external_index) == 3_547
assert len(main_train) + len(external_index) == 23_218
assert external_index["feature_year"].between(2021, 2022).all()
assert main_train["target_candidate_v2_pit_b"].isin([0.0, 1.0]).all()

accepted_preprocessing = PreprocessingPolicy(
    lower_quantile=0.01,
    upper_quantile=0.99,
    add_missing_indicators=True,
)
assert FROZEN_BLOCK_COMPARISONS == (("L",), ("L", "D"), ("L", "D", "R"))
assert [(fold.train_start, fold.train_end, fold.validation_start, fold.validation_end) for fold in MAIN_EXPANDING_WINDOW_FOLDS] == [
    (2011, 2013, 2015, 2015),
    (2011, 2014, 2016, 2016),
    (2011, 2015, 2017, 2017),
    (2011, 2016, 2018, 2018),
    (2011, 2017, 2019, 2019),
    (2011, 2018, 2020, 2020),
]

integrity = pd.DataFrame.from_dict(fingerprints, orient="index")
integrity["sha256_ok"] = True
integrity["shape_ok"] = True
show("Frozen input integrity", integrity[["rows", "columns", "MiB", "sha256_ok", "shape_ok"]])
print(
    f"\nAccepted sample verified: development N={len(main_train) + len(external_index):,}; "
    f"train 2011–2020 N={len(main_train):,}; external index 2021–2022 N={len(external_index):,}."
)
print("External target values/features were not loaded; 2023–2024 rows were not loaded.")
print(f"Runtime: pandas={pd.__version__}; numpy={np.__version__}; sklearn={sklearn.__version__}")



Frozen input integrity
                      rows  columns    MiB  sha256_ok  shape_ok
universe            103099       81  106.4       True      True
target               26917      802  169.3       True      True
x_t                  64901     1072  776.7       True      True
target_application   64901      832  383.7       True      True

Accepted sample verified: development N=23,218; train 2011–2020 N=19,671; external index 2021–2022 N=3,547.
External target values/features were not loaded; 2023–2024 rows were not loaded.
Runtime: pandas=3.0.3; numpy=2.4.4; sklearn=1.8.0


## 1. Selection path: target availability and `X_t` availability

Selection bias is assessed against all eligible train company-years 2011–2020.
The accepted estimand remains conditional on both gates: target available and
`X_t` in `{available_core, partially_available}`. The audit does not redefine
that estimand or the sample.


In [2]:
selection_flow = pd.DataFrame(
    [
        ("eligible universe train 2011–2020", len(eligible_train), 100.0),
        ("accepted X-status gate", int(eligible_train["x_usable"].sum()), 100 * eligible_train["x_usable"].mean()),
        ("target available (irrespective of X)", int(eligible_train["target_available"].sum()), 100 * eligible_train["target_available"].mean()),
        ("accepted main supervised train", len(main_train), 100 * eligible_train["main_selected"].mean()),
    ],
    columns=["stage", "n", "pct_of_eligible_train"],
).set_index("stage")

x_target_counts = pd.crosstab(
    eligible_train["x_t_status"], eligible_train["target_available"]
).rename(columns={0: "target_unavailable_n", 1: "target_available_n"})
x_target_counts["total_n"] = x_target_counts.sum(axis=1)
x_target_counts["target_available_pct"] = (
    100 * x_target_counts["target_available_n"] / x_target_counts["total_n"]
)
x_target_counts = x_target_counts.sort_values("total_n", ascending=False)

target_status_by_x = pd.crosstab(
    eligible_train["x_t_status"], eligible_train["target_status"]
)

show("Selection flow", selection_flow.round(2))
show("X_t status versus target availability", x_target_counts.round(2))
show("Detailed target status by X_t status", target_status_by_x)



Selection flow
                                          n  pct_of_eligible_train
stage                                                             
eligible universe train 2011–2020     47938                 100.00
accepted X-status gate                43175                  90.06
target available (irrespective of X)  19784                  41.27
accepted main supervised train        19671                  41.03

X_t status versus target availability
target_available        target_unavailable_n  target_available_n  total_n  target_available_pct
x_t_status                                                                                     
available_core                         18463               19159    37622                 50.92
partially_available                     5041                 512     5553                  9.22
not_available_non_xbrl                  3665                 103     3768                  2.73
missing                                  641                   

Target availability is not independent of `X_t` availability. It is 50.9% for
`available_core`, 9.2% for `partially_available`, 2.7% for non-XBRL and below
2% for missing/ambiguous `X_t`. This is evidence of a structured observation
process, not evidence that the accepted sample should be changed.

## 2. Main imputed sample versus complete cases

Complete cases are computed separately for L, L+D and L+D+R. They remain
robustness samples only; the main three-block comparison always uses the same
19,671 train observations with imputation.


In [3]:
complete_case_rows = []
complete_masks = {}
for blocks in FROZEN_BLOCK_COMPARISONS:
    label = "+".join(blocks)
    columns = [f"{feature}_value" for feature in features_for_blocks(blocks)]
    complete = main_train[columns].notna().all(axis=1)
    complete_masks[label] = complete
    subset = main_train.loc[complete]
    complete_case_rows.append(
        {
            "sample": f"complete-case {label}",
            "n": len(subset),
            "retained_pct": 100 * complete.mean(),
            "positive_n": int(subset["target_candidate_v2_pit_b"].sum()),
            "positive_rate_pct": 100 * subset["target_candidate_v2_pit_b"].mean(),
            "median_log_assets_t": subset["log_assets_t_value"].median(),
        }
    )
complete_case_summary = pd.DataFrame(
    [
        {
            "sample": "main imputed C",
            "n": len(main_train),
            "retained_pct": 100.0,
            "positive_n": int(main_train["target_candidate_v2_pit_b"].sum()),
            "positive_rate_pct": 100 * main_train["target_candidate_v2_pit_b"].mean(),
            "median_log_assets_t": main_train["log_assets_t_value"].median(),
        },
        *complete_case_rows,
    ]
).set_index("sample")

observed_size = eligible_train["log_assets_t_value"].notna()
eligible_train["size_quartile"] = "missing"
eligible_train.loc[observed_size, "size_quartile"] = pd.qcut(
    eligible_train.loc[observed_size, "log_assets_t_value"],
    q=4,
    labels=["Q1 smallest", "Q2", "Q3", "Q4 largest"],
).astype(str)
main_train["size_quartile"] = eligible_train.loc[main_train.index, "size_quartile"]
main_train["complete_LDR"] = complete_masks["L+D+R"].astype(int)

cc_by_year = main_train.groupby("feature_year", sort=True).agg(
    main_n=("complete_LDR", "size"),
    complete_LDR_n=("complete_LDR", "sum"),
    complete_LDR_pct=("complete_LDR", lambda x: 100 * x.mean()),
)
cc_by_sector = main_train.groupby("research_sector", sort=False).agg(
    main_n=("complete_LDR", "size"),
    complete_LDR_n=("complete_LDR", "sum"),
    complete_LDR_pct=("complete_LDR", lambda x: 100 * x.mean()),
).sort_values("main_n", ascending=False)
cc_by_size = main_train.groupby("size_quartile", sort=True).agg(
    main_n=("complete_LDR", "size"),
    complete_LDR_n=("complete_LDR", "sum"),
    complete_LDR_pct=("complete_LDR", lambda x: 100 * x.mean()),
)
cc_by_target = main_train.groupby("target_candidate_v2_pit_b").agg(
    main_n=("complete_LDR", "size"),
    complete_LDR_n=("complete_LDR", "sum"),
    complete_LDR_pct=("complete_LDR", lambda x: 100 * x.mean()),
).rename(index={0.0: "negative", 1.0: "positive"})

show("Main imputed sample versus block-specific complete cases", complete_case_summary.round(3))
show("L+D+R complete-case retention by year", cc_by_year.round(2))
show("L+D+R complete-case retention by sector", cc_by_sector.round(2))
show("L+D+R complete-case retention by size", cc_by_size.round(2))
show("L+D+R complete-case retention by target class", cc_by_target.round(2))



Main imputed sample versus block-specific complete cases
                         n  retained_pct  positive_n  positive_rate_pct  median_log_assets_t
sample                                                                                      
main imputed C       19671       100.000        3623             18.418               19.754
complete-case L      19159        97.397        3508             18.310               19.780
complete-case L+D    16962        86.228        3206             18.901               19.634
complete-case L+D+R  15654        79.579        2869             18.328               19.695

L+D+R complete-case retention by year
              main_n  complete_LDR_n  complete_LDR_pct
feature_year                                          
2011            1907            1753             91.92
2012            2311            1790             77.46
2013            2393            1780             74.38
2014            2300            1854             80.61
2015           

L+D+R complete cases retain only 79.6% of the accepted train sample. Similar
aggregate target balance does not establish exchangeability: retention differs
over time, sector and size, while sparse subgroups have materially different
target rates. Complete-case results therefore cannot replace the main imputed
estimand.

## 3. Missing indicators, sparse observations and planned ablations

Variant B removes indicator columns but does not remove observations. It is a
representation ablation on exactly the same rows, not a competing sample policy.
The table below is descriptive only and is not model performance.


In [4]:
representation = pd.DataFrame(
    [
        ("C main", len(main_train), 17, 17, "p1/p99 + median + scaler"),
        ("B ablation", len(main_train), 17, 0, "p1/p99 + median + scaler"),
        ("complete-case L+D+R robustness", int(complete_masks["L+D+R"].sum()), 17, 0, "row restriction; block-specific"),
        ("no-winsorization robustness", len(main_train), 17, 17, "median + scaler"),
    ],
    columns=["variant", "train_rows", "financial_columns", "indicator_columns", "other_operations"],
).set_index("variant")

indicator_rows = []
for feature, value_column in zip(FEATURES, VALUE_COLUMNS, strict=True):
    missing = main_train[value_column].isna()
    indicator_rows.append(
        {
            "feature": feature,
            "missing_n": int(missing.sum()),
            "missing_pct": 100 * missing.mean(),
            "positive_rate_missing_pct": 100 * main_train.loc[missing, "target_candidate_v2_pit_b"].mean(),
            "positive_rate_observed_pct": 100 * main_train.loc[~missing, "target_candidate_v2_pit_b"].mean(),
        }
    )
indicator_audit = pd.DataFrame(indicator_rows).set_index("feature")
indicator_audit["missing_minus_observed_pp"] = (
    indicator_audit["positive_rate_missing_pct"]
    - indicator_audit["positive_rate_observed_pct"]
)
indicator_audit = indicator_audit.sort_values(
    "missing_minus_observed_pp", key=lambda x: x.abs(), ascending=False
)

sparsity_bins = [-1, 4, 10, 13, 16, 17]
sparsity_labels = ["0–4", "5–10", "11–13", "14–16", "17"]
eligible_train["sparsity_band"] = pd.cut(
    eligible_train["available_feature_count"],
    bins=sparsity_bins,
    labels=sparsity_labels,
)
main_train["sparsity_band"] = eligible_train.loc[main_train.index, "sparsity_band"]
x_usable_train = eligible_train.loc[eligible_train["x_usable"].eq(1)].copy()
sparse_availability = x_usable_train.groupby("sparsity_band", observed=False).agg(
    candidate_n=("target_available", "size"),
    target_available_n=("target_available", "sum"),
    target_available_pct=("target_available", lambda x: 100 * x.mean()),
)
sparse_target = main_train.groupby("sparsity_band", observed=False).agg(
    supervised_n=("target_candidate_v2_pit_b", "size"),
    positive_n=("target_candidate_v2_pit_b", "sum"),
    positive_rate_pct=("target_candidate_v2_pit_b", lambda x: 100 * x.mean()),
)
sparse_audit = sparse_availability.join(sparse_target)

show("C/B/robustness row and feature schemas", representation)
show("Missing-indicator association with target in train only", indicator_audit.round(2))
show("Sparse observations: availability and target composition", sparse_audit.round(2))
print(
    f"\nVery sparse (<=10/17 observed) in main train: "
    f"{int(main_train['available_feature_count'].le(10).sum()):,} "
    f"({100 * main_train['available_feature_count'].le(10).mean():.2f}%)."
)
print("Large deltas based on very small missing_n must not be interpreted as stable effects.")



C/B/robustness row and feature schemas
                                train_rows  financial_columns  indicator_columns                 other_operations
variant                                                                                                          
C main                               19671                 17                 17         p1/p99 + median + scaler
B ablation                           19671                 17                  0         p1/p99 + median + scaler
complete-case L+D+R robustness       15654                 17                  0  row restriction; block-specific
no-winsorization robustness          19671                 17                 17                  median + scaler

Missing-indicator association with target in train only
                                missing_n  missing_pct  positive_rate_missing_pct  positive_rate_observed_pct  missing_minus_observed_pp
feature                                                                           

Missing indicators preserve potentially informative missingness patterns; B tests
whether that representation matters. Neither C nor B corrects missing-target or
non-XBRL selection. Sparse rows remain in the main matrix by design, and their
indicator patterns make their information limitation explicit.

## 4. Year, sector, size and XBRL availability

Rates below use the eligible train universe as denominator. `main_inclusion`
means simultaneous target availability and accepted `X_t` status.


In [5]:
def selection_rate_table(group: str) -> pd.DataFrame:
    return eligible_train.groupby(group, dropna=False, sort=True).agg(
        eligible_n=("target_available", "size"),
        target_available_n=("target_available", "sum"),
        target_available_pct=("target_available", lambda x: 100 * x.mean()),
        x_usable_n=("x_usable", "sum"),
        x_usable_pct=("x_usable", lambda x: 100 * x.mean()),
        main_inclusion_n=("main_selected", "sum"),
        main_inclusion_pct=("main_selected", lambda x: 100 * x.mean()),
    )

by_year = selection_rate_table("feature_year")
by_sector = selection_rate_table("research_sector").sort_values("eligible_n", ascending=False)
by_size = selection_rate_table("size_quartile")

eligible_train["xbrl_availability"] = np.select(
    [
        eligible_train["statement_scope_xbrl_available"].eq(True),
        eligible_train["xbrl_submission_available"].eq(True),
    ],
    ["statement_scope_available", "submission_only_scope_unavailable"],
    default="no_xbrl_submission",
)
by_xbrl = selection_rate_table("xbrl_availability")

non_xbrl = eligible_train.loc[
    eligible_train["x_t_status"].eq("not_available_non_xbrl")
].copy()
non_xbrl_target_available = non_xbrl.loc[non_xbrl["target_available"].eq(1)]
non_xbrl_summary = pd.Series(
    {
        "eligible_train_non_xbrl_n": len(non_xbrl),
        "target_available_non_xbrl_n": len(non_xbrl_target_available),
        "target_available_non_xbrl_pct": 100 * non_xbrl["target_available"].mean(),
        "positive_n_where_target_available": int(non_xbrl_target_available["target_candidate_v2_pit_b"].sum()),
        "positive_rate_where_target_available_pct": 100 * non_xbrl_target_available["target_candidate_v2_pit_b"].mean(),
        "rows_entering_model_matrix": 0,
    }
)

show("Selection by feature year", by_year.round(2))
show("Selection by research sector", by_sector.round(2))
show("Selection by time-t size", by_size.round(2))
show("Selection by XBRL availability", by_xbrl.round(2))
show("Non-XBRL observations outside the model matrix", non_xbrl_summary.round(2))



Selection by feature year
              eligible_n  target_available_n  target_available_pct  x_usable_n  x_usable_pct  main_inclusion_n  main_inclusion_pct
feature_year                                                                                                                      
2011                5662                1926                 34.02        3979         70.28              1907               33.68
2012                5364                2317                 43.20        4806         89.60              2311               43.08
2013                5262                2397                 45.55        4817         91.54              2393               45.48
2014                5148                2301                 44.70        4747         92.21              2300               44.68
2015                4851                2245                 46.28        4491         92.58              2241               46.20
2016                4592                2203            

Observed selection gradients are material. Target availability varies by year,
sector and size; missing size is itself a strong availability marker. All 3,768
non-XBRL train rows remain outside the model matrix, including 103 with an
available target. Imputation cannot manufacture their absent financial features,
and target-availability IPW cannot repair this separate first-stage support gap.

## 5. Target-availability IPW: pre-specified diagnostic model

### Estimand and denominator

IPW is evaluated only inside the already accepted `X_t` gate. The denominator is
all 43,175 eligible train rows with `x_t_status` in
`{available_core, partially_available}`; the selection event is
`target_status == available` (`N=19,671`). Thus IPW addresses target availability
only. It does not claim to recover non-XBRL, missing or ambiguous `X_t` rows.

### Propensity specification fixed before predictive modeling

- response: binary target availability, never the target label;
- time-t categorical covariates: feature year, research sector, `x_t_status`,
  statement-scope XBRL status, registrant role, XBRL submission/scope flags and
  joint-filing flag;
- time-t numeric covariates: `log_assets_t` and number of observed frozen
  financial features; median imputation, a size-missing indicator and scaling are
  internal to this diagnostic propensity pipeline;
- estimator: L2 logistic regression, `C=1`, no class reweighting, fixed
  `random_state=20260818`;
- no target components, t+1 financial values, target-availability reasons or
  predictive-model outputs are covariates.

This is a feasibility diagnostic on 2011–2020, not a candidate chosen by target
prediction performance. If IPW were admissible later, its propensity preprocessing
and fit would be repeated inside each temporal training fold only.


In [6]:
ipw_population = x_usable_train.copy()
ipw_population["feature_year_cat"] = ipw_population["feature_year"].astype(str)
categorical_covariates = [
    "feature_year_cat",
    "research_sector",
    "x_t_status",
    "statement_scope_xbrl_status",
    "registrant_role_resolved",
    "xbrl_submission_available",
    "statement_scope_xbrl_available",
    "joint_filing_flag",
]
numeric_covariates = ["log_assets_t_value", "available_feature_count"]
propensity_covariates = categorical_covariates + numeric_covariates
assert "target_candidate_v2_pit_b" not in propensity_covariates
assert not any(column.startswith("target_") for column in propensity_covariates)
for column in categorical_covariates:
    ipw_population[column] = (
        ipw_population[column].astype("string").fillna("__MISSING__")
    )

propensity_design = ColumnTransformer(
    [
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", drop=None),
            categorical_covariates,
        ),
        (
            "numeric",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_covariates,
        ),
    ],
    remainder="drop",
)
propensity_model = Pipeline(
    [
        ("design", propensity_design),
        (
            "logit",
            LogisticRegression(
                C=1.0,
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=2_000,
                class_weight=None,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)
propensity_model.fit(
    ipw_population[propensity_covariates],
    ipw_population["target_available"],
)
ipw_population["propensity"] = propensity_model.predict_proba(
    ipw_population[propensity_covariates]
)[:, 1]
selected_ipw = ipw_population.loc[ipw_population["target_available"].eq(1)].copy()
assert len(ipw_population) == 43_175
assert len(selected_ipw) == 19_671

def quantile_summary(values: pd.Series) -> pd.Series:
    return pd.Series(
        {
            "n": values.size,
            "min": values.min(),
            "p0.1": values.quantile(0.001),
            "p1": values.quantile(0.01),
            "p5": values.quantile(0.05),
            "median": values.median(),
            "p95": values.quantile(0.95),
            "p99": values.quantile(0.99),
            "p99.9": values.quantile(0.999),
            "max": values.max(),
        }
    )

propensity_summary = pd.DataFrame(
    {
        "all_X_usable": quantile_summary(ipw_population["propensity"]),
        "target_unavailable": quantile_summary(ipw_population.loc[ipw_population["target_available"].eq(0), "propensity"]),
        "target_available": quantile_summary(selected_ipw["propensity"]),
    }
).T

score_edges = np.array([0, .05, .10, .20, .30, .40, .50, .60, .70, .80, .90, .95, 1.0])
ipw_population["propensity_bin"] = pd.cut(
    ipw_population["propensity"], score_edges, include_lowest=True
)
score_histogram = pd.crosstab(
    ipw_population["propensity_bin"], ipw_population["target_available"]
).rename(columns={0: "target_unavailable_n", 1: "target_available_n"})

ipw_population["propensity_decile"] = pd.qcut(
    ipw_population["propensity"], 10, duplicates="drop"
)
calibration = ipw_population.groupby("propensity_decile", observed=False).agg(
    n=("target_available", "size"),
    predicted_availability=("propensity", "mean"),
    observed_availability=("target_available", "mean"),
)
positivity = pd.Series(
    {
        "candidate_population_n": len(ipw_population),
        "candidate_propensity_below_0.05_n": int(ipw_population["propensity"].lt(0.05).sum()),
        "candidate_propensity_below_0.05_pct": 100 * ipw_population["propensity"].lt(0.05).mean(),
        "selected_propensity_below_0.05_n": int(selected_ipw["propensity"].lt(0.05).sum()),
        "selected_propensity_below_0.05_pct": 100 * selected_ipw["propensity"].lt(0.05).mean(),
        "selected_min_propensity": selected_ipw["propensity"].min(),
        "candidate_propensity_above_0.95_n": int(ipw_population["propensity"].gt(0.95).sum()),
    }
)

show("Propensity-score distribution", propensity_summary.round(6))
show("Propensity-score histogram", score_histogram)
show("Propensity calibration by score decile", calibration.round(4))
show("Positivity diagnostics", positivity.round(6))



Propensity-score distribution
                          n       min      p0.1        p1        p5    median       p95       p99     p99.9       max
all_X_usable        43175.0  0.000115  0.000584  0.002529  0.014318  0.423102  0.877666  0.916268  0.945554  0.964523
target_unavailable  23504.0  0.000115  0.000428  0.001794  0.007315  0.216789  0.790990  0.876300  0.923485  0.960195
target_available    19671.0  0.000829  0.003218  0.011765  0.172749  0.757499  0.898136  0.927565  0.949663  0.964523

Propensity-score histogram
target_available  target_unavailable_n  target_available_n
propensity_bin                                            
(-0.001, 0.05]                    4166                 358
(0.05, 0.1]                       2473                 202
(0.1, 0.2]                        4329                 665
(0.2, 0.3]                        4347                1018
(0.3, 0.4]                        2491                 927
(0.4, 0.5]                        1324                 8

The low-score region is not empty noise: 358 selected observations have estimated
propensity below 5%, including partially available and very sparse rows. Positivity
must therefore be assessed through the resulting weights and effective sample size,
not merely through the overall selection rate.


In [7]:
selection_rate = ipw_population["target_available"].mean()
selected_ipw["unstabilized_weight"] = 1.0 / selected_ipw["propensity"]
selected_ipw["stabilized_weight"] = selection_rate / selected_ipw["propensity"]
truncation_p01, truncation_p99 = selected_ipw["stabilized_weight"].quantile([0.01, 0.99])
selected_ipw["stabilized_p1_p99_weight"] = selected_ipw["stabilized_weight"].clip(
    truncation_p01, truncation_p99
)

def effective_sample_size(weights: pd.Series) -> float:
    return float(weights.sum() ** 2 / weights.pow(2).sum())

weight_rows = []
for label, column in (
    ("unstabilized 1/p", "unstabilized_weight"),
    ("stabilized", "stabilized_weight"),
    ("stabilized + p1/p99 truncation", "stabilized_p1_p99_weight"),
):
    weights = selected_ipw[column]
    ess = effective_sample_size(weights)
    weight_rows.append(
        {
            "weight_policy": label,
            "mean": weights.mean(),
            "sd": weights.std(ddof=1),
            "p95": weights.quantile(0.95),
            "p99": weights.quantile(0.99),
            "p99.9": weights.quantile(0.999),
            "max": weights.max(),
            "ESS": ess,
            "ESS_pct_of_selected": 100 * ess / len(weights),
        }
    )
weight_diagnostics = pd.DataFrame(weight_rows).set_index("weight_policy")

def weighted_distribution(
    data: pd.DataFrame, column: str, weight_column: str | None = None
) -> pd.Series:
    if weight_column is None:
        values = data.groupby(column, dropna=False).size().astype(float)
    else:
        values = data.groupby(column, dropna=False)[weight_column].sum()
    return values / values.sum()

balance_rows = []
for column in ("feature_year", "research_sector", "x_t_status", "sparsity_band"):
    candidate = weighted_distribution(ipw_population, column)
    unweighted = weighted_distribution(selected_ipw, column)
    stabilized = weighted_distribution(selected_ipw, column, "stabilized_weight")
    truncated = weighted_distribution(selected_ipw, column, "stabilized_p1_p99_weight")
    index = candidate.index.union(unweighted.index).union(stabilized.index).union(truncated.index)
    def total_variation(other: pd.Series) -> float:
        return float(
            0.5
            * (
                candidate.reindex(index, fill_value=0)
                - other.reindex(index, fill_value=0)
            ).abs().sum()
        )
    balance_rows.append(
        {
            "dimension": column,
            "unweighted_TV": total_variation(unweighted),
            "stabilized_IPW_TV": total_variation(stabilized),
            "truncated_IPW_TV": total_variation(truncated),
        }
    )
balance_audit = pd.DataFrame(balance_rows).set_index("dimension")

show("IPW weight distribution and effective sample size", weight_diagnostics.round(3))
show(
    "Candidate p1/p99 truncation bounds for stabilized weights",
    pd.Series({"p1_lower": truncation_p01, "p99_upper": truncation_p99}).round(6),
)
show("Distribution balance: total-variation distance to candidate population", balance_audit.round(4))



IPW weight distribution and effective sample size
                                 mean      sd    p95     p99    p99.9       max       ESS  ESS_pct_of_selected
weight_policy                                                                                                 
unstabilized 1/p                4.168  25.326  5.789  84.995  310.793  1206.694   518.821                2.637
stabilized                      1.899  11.539  2.637  38.725  141.601   549.783   518.821                2.637
stabilized + p1/p99 truncation  1.401   4.297  2.637  38.701   38.725    38.725  1889.230                9.604

Candidate p1/p99 truncation bounds for stabilized weights
p1_lower      0.491190
p99_upper    38.724828

Distribution balance: total-variation distance to candidate population
                 unweighted_TV  stabilized_IPW_TV  truncated_IPW_TV
dimension                                                          
feature_year            0.0501             0.2767            0.2105
research_sect

## 6. IPW decision

IPW is **not methodologically reliable enough to use in the current pipeline**:

- 4,524/43,175 candidate rows (10.5%) have propensity below 5%; 358 of them
  nevertheless have an available target;
- minimum propensity among selected rows is about 0.083%, producing maximum raw
  weight about 1,207 and stabilized weight about 550;
- untruncated ESS is only about 519, or 2.6% of the 19,671 selected rows;
- pre-specified stabilization plus p1/p99 truncation raises ESS only to about
  1,889 (9.6%) and materially changes the estimand;
- weighted balance is not uniformly improved: year, `x_t_status` and sparsity
  remain unstable or worsen;
- target availability also depends on t+1 filing/content processes that cannot be
  represented using the required time-t-only covariates, making the necessary MAR
  assumption implausible and untestable;
- this IPW estimand cannot address non-XBRL or otherwise unavailable `X_t` rows.

This rejection is based only on propensity/positivity diagnostics, never on
predictive target performance. The p1/p99 stabilized policy is documented as the
pre-specified candidate that was assessed, but **no IPW-weighted model metric is
authorized in the frozen pipeline**. Revisiting IPW would require a new version,
new covariates with defensible time-t availability, and a new positivity audit.

## 7. Final supervised ML specification

### Supervised sample

- `target_status == available`;
- `x_t_status in {available_core, partially_available}`;
- development `N=23,218`; train 2011–2020 `N=19,671`; external validation
  2021–2022 `N=3,547`;
- estimand is explicitly conditional on these observation gates; claims do not
  extend to non-XBRL or target-unavailable company-years.

### Preprocessing and blocks

- C: fold/train-only p1/p99 winsorization, fold/train median imputation, one
  missing indicator per feature, StandardScaler for financial features only;
  indicators remain binary;
- B without indicators is the mandatory ablation;
- complete-case and no-winsorization are robustness checks only;
- L, L+D and L+D+R use the same main rows; missing R never drops a main row.

### Temporal CV and groups

| Fold | Train | Embargo | Validation |
|---|---|---|---|
| 2015 | 2011–2013 | 2014 | 2015 |
| 2016 | 2011–2014 | 2015 | 2016 |
| 2017 | 2011–2015 | 2016 | 2017 |
| 2018 | 2011–2016 | 2017 | 2018 |
| 2019 | 2011–2017 | 2018 | 2019 |
| 2020 | 2011–2018 | 2019 | 2020 |

Every training row additionally satisfies
`target_available_at <= min(prediction_timestamp in validation fold)`.
Preprocessing is refit from zero inside each safe training partition; fold
validation is transform-only. Purged `economic_group_id` CV uses the same temporal
folds and remains robustness only. Group ID is metadata, never a predictor.

### Metric aggregation and inference

- primary ranking: pooled OOF PR-AUC across validation years 2015–2020;
- report fold PR-AUC plus arithmetic mean and sample SD (`ddof=1`);
- secondary metrics cannot change primary ranking;
- inference: 2,000 bootstrap replicates resampling `economic_group_id` clusters
  with replacement and preserving every row/multiplicity in each sampled group;
- 95% percentile CI (`2.5%`, `97.5%`), fixed seed `20260818`; row-level bootstrap
  is never the primary CI. A degenerate single-class draw is deterministically
  redrawn and the rejection count reported.

### Selection-bias robustness

- mandatory: C versus B, block-specific complete cases, no-winsorization and
  purged group CV, all labeled as ablation/robustness rather than alternate
  primary rankings;
- always report sample retention and composition by year, sector, time-t size,
  `x_t_status`, XBRL availability, sparsity and target class;
- non-XBRL rows remain outside the model matrix and the scope limitation is
  explicit;
- IPW policy: diagnostic evaluated and rejected for this version because of poor
  positivity, extreme weights, low ESS and an unsupported MAR assumption.

### External validation

2021–2022 remains a one-shot no-tune external development validation. It may be
opened only after model family, hyperparameters, training budget/seeds,
preprocessing, feature block, threshold/calibration, primary ranking, refit and
inference policies are locked. After opening, failure means no-go or a clearly new
pipeline version; those years cannot remain an independent validation for the
revised version. Test 2023–2024 remains unopened until the entire model-stage
specification is frozen.

## Freeze-gate verdict

The accepted pipeline has a coherent conditional estimand, leakage-safe temporal
CV, fold-only preprocessing, a fixed ranking rule, cluster-aware inference and
explicit limitations where IPW cannot credibly repair selection. No blocking
pipeline-design ambiguity remains before model-family preregistration.

# SUPERVISED ML PIPELINE READY TO FREEZE

This is a gate verdict only. **The pipeline has not been automatically frozen.**
No predictive model, CV score or test result was produced.
